In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("synysterjeet/food-classification")

print("Path to dataset files:", path)

In [5]:
import os
import pickle
from skimage.io import imread
from skimage.transform import resize
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score


In [9]:
input_dir = r'PATH_TO_DATASET/dataset/train'  # Update this path
categories = []
for file in os.listdir(input_dir):
    categories.append(file)
data = []
labels = []
for category_idx, category in enumerate(categories):
    for file in os.listdir(os.path.join(input_dir, category)):
        img_path = os.path.join(input_dir, category, file)
        img = imread(img_path)
        if img.shape[-1] == 4:
            img = img[:, :, :3]
        if img.ndim == 2:
            img = np.stack([img]*3, axis=-1)
        img = resize(img, (15, 15, 3))
        data.append(img.flatten())
        labels.append(category_idx)
    print(category_idx)
data = np.asarray(data)
labels = np.asarray(labels)

In [25]:
# Filter out inconsistent image sizes
clean_data = []
clean_labels = []
for img, label in zip(data, labels):
    if img.shape[0] == 675:
        clean_data.append(img)
        clean_labels.append(label)
    else:
        print(f"Skipping image with shape: {img.shape}")
data = np.array(clean_data)
labels = np.array(clean_labels)
np.save("processed_data.npy", data)
np.save("processed_labels.npy", labels)
print("Saved cleaned preprocessed data and labels.")

In [31]:
x_train, x_test, y_train, y_test = train_test_split(data, labels, test_size=0.2, shuffle=True, stratify=labels)

In [33]:
classifier = SVC()
parameters = [{'gamma': [0.01], 'C': [100]}]
grid_search = GridSearchCV(classifier, parameters)
grid_search.fit(x_train, y_train)

In [34]:
best_estimator = grid_search.best_estimator_
y_prediction = best_estimator.predict(x_test)
score = accuracy_score(y_prediction, y_test)
print(f'{score*100}% of samples were correctly classified')

In [42]:
pickle.dump(best_estimator, open('./490Image.pkl', 'wb'))

In [7]:
categories = sorted(os.listdir(input_dir))
category_dict = {idx: category for idx, category in enumerate(categories)}
np.save('category_dict.npy', category_dict)